# Run jaxions 2D cylindrical simulations

The program works with very few options, it is likely to segfault if you run it with others. 

The options are specified in the caxion23loops.py functions

caxion3d currently only works in the CPU (other propagators are standard 2D without cylindrical boundaries)

Boundary conditions are 16 points sponges so make sure you only measure quantities where boundary effects are causally disconnected. 

output data are moved to folder /data/out... make sure you create the /data directory!

some data for ICs is saved in /aux, make sure you have it too!

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pyaxions import jaxions as pa
import caxion23loops as ca

# Preparations 

In [ ]:
# create data dirs
!mkdir data
!mkdir aux

In [ ]:
# build an interpolation function to speed up ICs
ca.buildf1f2(ninterp=10000)

In [ ]:
# help(ca.simu)

# Run

In [ ]:
# run a simulation in the CPU with 256, 4 MPI ranks, 1 OMP thread, laplacian with Ng neighbours
ca.simu(R=128.5,msa=0.5,N=256,Ng=2,
        Np=4,omp=1,
        options=' --p2DmapYZ ', # use = '' if maps not wanted
        plota=True,n_save=200,gpu=False,verb=0)

# Read

data is saved according to N,msa and Ng (different values of R would take the same name so watch out!)

In [ ]:
mf = pa.fm('data/out256-500-2/',True)

In [ ]:
# # the field phi(z=0) as a function of rho is saved automatically
for it in [0,10,20,30]:
    fe = np.reshape(pa.gm(mf[it],'da/chunk/m'),(256,2))
    plt.plot(fe[:,0])
    plt.plot(fe[:,1])
# by symmetry, Im(phi(z=0)=0
# the string center is at Re(phi)=0

In [ ]:
# slices are also saved but if run with options=' --p2DmapYZ ',
fig,ax=plt.subplots(1,2,figsize=(12,5))
i = ax[0].imshow(pa.gm(mf[0],'mapptheta'),origin='lower',vmin=-np.pi,vmax=np.pi,cmap=pa.thetacmap)
pa.colorbar(i)
i = ax[1].imshow(pa.gm(mf[-1],'mapptheta'),origin='lower',vmin=-np.pi,vmax=np.pi,cmap=pa.thetacmap)
pa.colorbar(i)

In [ ]:
# list measurement files
mf = pa.fm('data/out256-500-2/')
# build the radius as a function of t and estimate of gamma
t,r,g = ca.buildr(mf)
plt.plot(t,r)
plt.plot(t,r[1]*np.cos(t/r[1]))